# Phase 2: DuckDB 查詢引擎測試

**目標**: 學習如何使用 DuckDB 查詢 Parquet 檔案

**測試資料**: `data/processed/2025_station_hour_processed.parquet`

**學習重點**:
1. 建立 DuckDB 連線
2. 直接查詢 Parquet 檔案(不載入到記憶體)
3. 使用 VIEW 簡化查詢
4. 執行聚合查詢(GROUP BY, AVG, COUNT)
5. 轉換為 pandas DataFrame
6. 效能比較: DuckDB vs pandas

---

## 步驟 0: 環境準備

In [22]:
# load packages
import duckdb
import pandas as pd # convert DuckDB to df
import time
from pathlib import Path

# setup path
DATA_DIR = Path("../data/processed")
# / The operator is "redefined" by the Path object as path concatenation.
PARQUET_FILE = DATA_DIR / "2019_station_hour_processed.parquet"

print(DATA_DIR)
print(PARQUET_FILE)
if PARQUET_FILE.exists():
    print(f"find parquet data: {PARQUET_FILE}")
    print(f"parquet file size: {PARQUET_FILE.stat().st_size / 1024 / 1024:.2f} MB")
else:
    print(f"file not found. check the file path")



../data/processed
../data/processed/2019_station_hour_processed.parquet
find parquet data: ../data/processed/2019_station_hour_processed.parquet
parquet file size: 15.33 MB


---

## 步驟 1: 建立 DuckDB 連線

**重點**: DuckDB 是嵌入式資料庫,不需要啟動伺服器

In [23]:
# Setup DuckDB connection
# db_path=":memory:" means using memory mode (data will disappear after closing)
# use db_path="mydb.db" to create a persistent database.
conn = duckdb.connect(database=":memory")



---

## 步驟 2: 直接查詢 Parquet 檔案(最基礎方式)

**重點**: DuckDB 可以直接查詢 Parquet,不需要先載入

In [24]:
# The read_parquet() used in the SQL query is a function of DuckDB
query1 = f"""
SELECT * FROM read_parquet('{PARQUET_FILE}')
"""

result1 = conn.execute(query1)

# the query result is a duckdb object
print(result1)

# use pandas to convert to dataframe
df1 = result1.df()

df1.head()


,station,district,route,direction,type,year,hour,month,lanes,length,...,sd_speed,median_occup,avg_occup,sd_occup,days_observed,county,state_pm,abs_pm,latitude,longitude
0,1201054,12,133,S,ML,2019,0,January,4,1.285,...,0.396782,0.0030,0.002962,0.000364,13,59.0,9,8.991,33.662,-117.755
1,1201054,12,133,S,ML,2019,1,January,4,1.285,...,0.851846,0.0020,0.002085,0.000581,13,59.0,9,8.991,33.662,-117.755
2,1201054,12,133,S,ML,2019,2,January,4,1.285,...,0.399037,0.0016,0.001669,0.000417,13,59.0,9,8.991,33.662,-117.755
3,1201054,12,133,S,ML,2019,3,January,4,1.285,...,0.476364,0.0028,0.002723,0.000407,13,59.0,9,8.991,33.662,-117.755
4,1201054,12,133,S,ML,2019,4,January,4,1.285,...,0.606014,0.0076,0.007579,0.000717,14,59.0,9,8.991,33.662,-117.755


In [37]:
# more complicated query, still use read_parquet()
query2 = f"""
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT route) as total_routes,
    COUNT(DISTINCT station) as total_stations
FROM read_parquet('{PARQUET_FILE}')
"""

result2 = conn.execute(query2)

df2 = result2.df()
df2

,total_rows,total_routes,total_stations
0,230818,11,1259


In [13]:
# 方式 1: 直接使用 read_parquet() 函數
# 這是最基礎的方式,適合一次性查詢
query = f"""
SELECT 
    COUNT(*) as total_rows,
    COUNT(DISTINCT route) as total_routes,
    COUNT(DISTINCT station) as total_stations
FROM read_parquet('{PARQUET_FILE}')
"""

# 執行查詢
result = conn.execute(query)

print(result)
# 轉換為 DataFrame
df_summary = result.df()

print("📊 2025 年資料摘要:")
print(df_summary)
print(f"\n總行數: {df_summary['total_rows'][0]:,}")
print(f"路線數: {df_summary['total_routes'][0]}")
print(f"測站數: {df_summary['total_stations'][0]}")

📊 2025 年資料摘要:
   total_rows  total_routes  total_stations
0      230818            11            1259

總行數: 230,818
路線數: 11
測站數: 1259


---

## 步驟 3: 建立 VIEW(推薦方式)

**重點**: VIEW 讓查詢更簡潔,不需要每次都寫 `read_parquet()`

In [8]:
# create VIEW - it does not load data into RAM; it only creates a "virtual pointer."
# VIEW will be faster when querying parquet

# CREATE a VIEW or REPLACE the existing VIEW
conn.execute(f"""
    CREATE OR REPLACE VIEW traffic_2025 AS 
    SELECT * FROM read_parquet('{PARQUET_FILE}')
""")

result = conn.execute(f"""
    SELECT * FROM traffic_2025
""")

result_df = result.df()
print(len(result_df))
result_df.head(10)

230818


,station,district,route,direction,type,year,hour,month,lanes,length,...,sd_speed,median_occup,avg_occup,sd_occup,days_observed,county,state_pm,abs_pm,latitude,longitude
0,1201066,12,133,N,ML,2025,0,January,3,0.67,...,0.391036,0.00450,0.004436,0.000779,11,59.0,9,8.991,33.662,-117.755
1,1201066,12,133,N,ML,2025,1,January,3,0.67,...,0.586399,0.00255,0.002492,0.000512,12,59.0,9,8.991,33.662,-117.755
2,1201066,12,133,N,ML,2025,2,January,3,0.67,...,0.347197,0.00200,0.002073,0.000403,11,59.0,9,8.991,33.662,-117.755
3,1201066,12,133,N,ML,2025,3,January,3,0.67,...,0.352480,0.00190,0.001958,0.000444,12,59.0,9,8.991,33.662,-117.755
4,1201066,12,133,N,ML,2025,4,January,3,0.67,...,0.443760,0.00380,0.003992,0.000368,13,59.0,9,8.991,33.662,-117.755
5,1201066,12,133,N,ML,2025,5,January,3,0.67,...,0.968544,0.01050,0.010392,0.001014,13,59.0,9,8.991,33.662,-117.755
6,1201066,12,133,N,ML,2025,6,January,3,0.67,...,1.536479,0.02870,0.027938,0.002423,13,59.0,9,8.991,33.662,-117.755
7,1201066,12,133,N,ML,2025,7,January,3,0.67,...,2.004418,0.05960,0.057246,0.008974,13,59.0,9,8.991,33.662,-117.755
8,1201066,12,133,N,ML,2025,8,January,3,0.67,...,2.092385,0.06490,0.062962,0.006643,13,59.0,9,8.991,33.662,-117.755
9,1201066,12,133,N,ML,2025,9,January,3,0.67,...,1.213017,0.05340,0.053023,0.004183,13,59.0,9,8.991,33.662,-117.755


### Combine Multiple parquet to a VIEW

In [ ]:
# Create a VIEW from multiple parquet files

# Method 1. Wildcard (*) - Read all parquet files in a directory
conn.execute(f"""
    CREATE OR REPLACE VIEW all_traffic as
    SELECT * FROM read_parquet('{DATA_DIR}/*_station_hour_processed.parquet')
""")


traffic_query_2023_2025 = f"""
    SELECT * FROM all_traffic
    WHERE year >=2020 AND year <= 2025
"""

traffic_2023_2025_df = conn.execute(all_traffic_query).df()
traffic_2023_2025_df.head(10)

,station,district,route,direction,type,year,hour,month,lanes,length,...,sd_speed,median_occup,avg_occup,sd_occup,days_observed,county,state_pm,abs_pm,latitude,longitude
0,1201054,12,133,S,ML,2020,0,January,4,1.285,...,0.402078,0.0030,0.002885,0.000800,13,59.0,9,8.991,33.662,-117.755
1,1201054,12,133,S,ML,2020,1,January,4,1.285,...,0.522077,0.0019,0.001831,0.000487,13,59.0,9,8.991,33.662,-117.755
2,1201054,12,133,S,ML,2020,2,January,4,1.285,...,0.533253,0.0020,0.001962,0.000559,13,59.0,9,8.991,33.662,-117.755
3,1201054,12,133,S,ML,2020,3,January,4,1.285,...,0.462208,0.0026,0.002483,0.000587,12,59.0,9,8.991,33.662,-117.755
4,1201054,12,133,S,ML,2020,4,January,4,1.285,...,0.749949,0.0080,0.008000,0.001073,12,59.0,9,8.991,33.662,-117.755


In [77]:
# Method 2. based on the selected filter
import duckdb
from pathlib import Path


selected_years = [2023, 2024, 2025]

file_list = [str(DATA_DIR / f"{year}_station_hour_processed.parquet") for year in selected_years]

selected_year_query = f"""
CREATE OR REPLACE VIEW selected_year_traffic AS
SELECT * FROM read_parquet({file_list})
"""

conn.execute(selected_year_query)


# select I-5 from selected year traffic
selected_year_route_5_traffic_query = f"""
    SELECT * FROM selected_year_traffic
    WHERE route = '5'
"""
selected_year_route_5_traffic_df = conn.execute(selected_year_route_5_traffic_query).df()
display(selected_year_route_5_traffic_df)


,station,district,route,direction,type,year,hour,month,lanes,length,...,sd_speed,median_occup,avg_occup,sd_occup,days_observed,county,state_pm,abs_pm,latitude,longitude
0,1204193,12,5,S,ML,2023,0,January,5,0.765,...,0.494515,0.00795,0.008608,0.002038,12,59,.64,72.835,33.405,-117.598
1,1204193,12,5,S,ML,2023,1,January,5,0.765,...,0.539079,0.00540,0.005942,0.001398,12,59,.64,72.835,33.405,-117.598
2,1204193,12,5,S,ML,2023,2,January,5,0.765,...,0.574984,0.00445,0.005175,0.001571,12,59,.64,72.835,33.405,-117.598
3,1204193,12,5,S,ML,2023,3,January,5,0.765,...,0.731126,0.00590,0.006275,0.001686,12,59,.64,72.835,33.405,-117.598
4,1204193,12,5,S,ML,2023,4,January,5,0.765,...,0.981032,0.01140,0.012167,0.002415,12,59,.64,72.835,33.405,-117.598
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231733,1221774,12,5,S,HV,2025,19,September,2,1.275,...,0.677476,0.02810,0.029438,0.004602,13,59,6.82,79.015,33.467,-117.671
231734,1221774,12,5,S,HV,2025,20,September,2,1.275,...,0.509902,0.02620,0.028054,0.004984,13,59,6.82,79.015,33.467,-117.671
231735,1221774,12,5,S,HV,2025,21,September,2,1.275,...,0.943058,0.02010,0.020900,0.004854,13,59,6.82,79.015,33.467,-117.671
231736,1221774,12,5,S,HV,2025,22,September,2,1.275,...,0.191485,0.01140,0.012338,0.002805,13,59,6.82,79.015,33.467,-117.671


---

## 步驟 4: 基礎查詢範例

### 4.1 查看資料結構

In [ ]:
# query: the first five row in year 2025

query = f"""
    SELECT * 
    FROM traffic_2025
    LIMIT 5
"""

result_df = conn.execute(query).df()
result_df

,station,district,route,direction,type,year,hour,month,lanes,length,...,sd_speed,median_occup,avg_occup,sd_occup,days_observed,county,state_pm,abs_pm,latitude,longitude
0,1201066,12,133,N,ML,2025,0,January,3,0.67,...,0.391036,0.00450,0.004436,0.000779,11,59.0,9,8.991,33.662,-117.755
1,1201066,12,133,N,ML,2025,1,January,3,0.67,...,0.586399,0.00255,0.002492,0.000512,12,59.0,9,8.991,33.662,-117.755
2,1201066,12,133,N,ML,2025,2,January,3,0.67,...,0.347197,0.00200,0.002073,0.000403,11,59.0,9,8.991,33.662,-117.755
3,1201066,12,133,N,ML,2025,3,January,3,0.67,...,0.352480,0.00190,0.001958,0.000444,12,59.0,9,8.991,33.662,-117.755
4,1201066,12,133,N,ML,2025,4,January,3,0.67,...,0.443760,0.00380,0.003992,0.000368,13,59.0,9,8.991,33.662,-117.755


### 4.2 查看可用的年份和月份

In [54]:
# Query: Check which years and months of data are available
query = """
SELECT 
    DISTINCT year, 
    month,
    COUNT(*) as record_count
FROM traffic_2025
GROUP BY year, month
ORDER BY year, month
"""

df_time = conn.execute(query).df()
display(df_time.head())
display(df_time)

,year,month,record_count
0,2025,April,25616
1,2025,August,26297
2,2025,February,24379
3,2025,January,24098
4,2025,July,26871


,year,month,record_count
0,2025,April,25616
1,2025,August,26297
2,2025,February,24379
3,2025,January,24098
4,2025,July,26871
5,2025,June,25849
6,2025,March,24987
7,2025,May,26686
8,2025,September,26035


---

## 步驟 5: 進階查詢範例

### 5.1 路線查詢(WHERE 過濾)

In [57]:
# Query: Retrieve all data for I-5 ML northbound 
query = """
SELECT 
    station,
    route,
    direction,
    type,
    month,
    hour,
    avg_flow,
    avg_speed,
    lanes
FROM traffic_2025
WHERE route = '5' 
  AND direction = 'N'
  AND type = 'ML'
ORDER BY month, hour
"""

df_i5_n_ml = conn.execute(query).df()
display(df_i5_n_ml)

,station,route,direction,type,month,hour,avg_flow,avg_speed,lanes
0,1212115,5,N,ML,April,0,1359.642857,67.150000,5
1,1210926,5,N,ML,April,0,712.785714,68.842857,5
2,1205341,5,N,ML,April,0,1312.000000,70.300000,6
3,1205562,5,N,ML,April,0,1654.071429,70.378571,6
4,1204328,5,N,ML,April,0,752.642857,70.850000,6
...,...,...,...,...,...,...,...,...,...
17292,1204453,5,N,ML,September,23,1383.833333,71.125000,7
17293,1205380,5,N,ML,September,23,2398.769231,70.484615,6
17294,1204472,5,N,ML,September,23,1179.461538,66.915385,5
17295,1204211,5,N,ML,September,23,927.692308,68.446154,5


### 5.2 聚合查詢(GROUP BY + 聚合函數)

In [58]:
# 查詢: 每條路線的平均流量和速度
query = """
SELECT 
    route,
    COUNT(*) as record_count,
    AVG(avg_flow) as mean_flow,
    AVG(avg_speed) as mean_speed,
    MEDIAN(avg_flow) as median_flow,
    MEDIAN(avg_speed) as median_speed
FROM traffic_2025
GROUP BY route
ORDER BY mean_flow DESC
"""

df_routes = conn.execute(query).df()
display(df_routes)

,route,record_count,mean_flow,mean_speed,median_flow,median_speed
0,91,19351,2775.247856,59.820119,1861.733333,63.308333
1,5,70224,2660.087013,62.063383,1345.861538,64.469231
2,57,21209,2455.755149,59.268351,1213.538462,63.708333
3,405,32512,2419.678767,62.083598,939.261364,64.716667
4,55,21834,2250.704466,59.366632,1166.923077,64.220000
5,22,19248,1986.355400,61.116993,1037.916667,64.346154
6,605,2232,1690.590829,62.677480,612.071429,64.764286
7,73,17413,1469.247190,65.818016,1154.307692,67.325000
8,133,5177,970.975709,65.742239,698.400000,66.450000
9,241,14400,832.915797,66.466364,615.881410,66.983974


### 5.3 使用專案的 DuckDBQueryEngine

**重點**: 測試你實作的 `src/pems/query.py`

In [4]:
# setup directory path
import sys
sys.path.insert(0, '../')  # Add the project root directory to the Python path

# import DuckDBQueryEngine
from src.pems.query import DuckDBQueryEngine

# 建立查詢引擎
engine = DuckDBQueryEngine(data_source="local")

print("DuckDBQueryEngine initiated")

INFO:src.pems.query:Initializing DuckDB query engine (source: local)
INFO:src.pems.query:✅ Loaded httpfs extension (S3-compatible storage)
INFO:src.pems.query:✅ Performance config: 4 threads, 2GB memory limit
INFO:src.pems.query:Setting up local Parquet data source...
INFO:src.pems.query:✅ Loaded 7 Parquet files (2,258,850 total rows) into 'traffic_data' view
INFO:src.pems.query:📅 Available years: [2019, 2020, 2021, 2022, 2023, 2024, 2025]
INFO:src.pems.query:✅ DuckDB engine initialized successfully


DuckDBQueryEngine initiated


In [69]:
# engine default all traffic parquet VIEW name: traffic_data
traffic_2020_2021_i5_df = engine.execute(f"""
    SELECT * 
    FROM traffic_data
    WHERE year >= 2020
      AND year <= 2021 
      AND route = 5
""").df()
display(traffic_2020_2021_i5_df)

# query_to_df function
traffic_2023_2025_sr91_df = engine.query_to_df(f"""
    SELECT * 
    FROM traffic_data
    WHERE year >= 2023
      AND year <= 2025 
      AND route = 91
""")
display(traffic_2023_2025_sr91_df)

,station,district,route,direction,type,year,hour,month,lanes,length,...,sd_speed,median_occup,avg_occup,sd_occup,days_observed,county,state_pm,abs_pm,latitude,longitude
0,1204193,12,5,S,ML,2020,0,January,5,0.765,...,2.026846,0.01290,0.013300,0.001195,12,59.0,.64,72.835,33.405,-117.598
1,1204193,12,5,S,ML,2020,1,January,5,0.765,...,3.708712,0.01160,0.011375,0.001826,12,59.0,.64,72.835,33.405,-117.598
2,1204193,12,5,S,ML,2020,2,January,5,0.765,...,3.135815,0.01080,0.010508,0.001403,12,59.0,.64,72.835,33.405,-117.598
3,1204193,12,5,S,ML,2020,3,January,5,0.765,...,4.023445,0.01250,0.012267,0.000926,12,59.0,.64,72.835,33.405,-117.598
4,1204193,12,5,S,ML,2020,4,January,5,0.765,...,5.512211,0.02355,0.022908,0.001943,12,59.0,.64,72.835,33.405,-117.598
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
197512,1222038,12,5,S,HV,2021,19,December,2,0.430,...,3.655802,0.04705,0.049710,0.010233,10,59.0,13.7,85.895,33.558,-117.673
197513,1222038,12,5,S,HV,2021,20,December,2,0.430,...,4.401124,0.04490,0.045080,0.010818,10,59.0,13.7,85.895,33.558,-117.673
197514,1222038,12,5,S,HV,2021,21,December,2,0.430,...,12.190889,0.03115,0.037280,0.021152,10,59.0,13.7,85.895,33.558,-117.673
197515,1222038,12,5,S,HV,2021,22,December,2,0.430,...,12.297827,0.02215,0.023660,0.014708,10,59.0,13.7,85.895,33.558,-117.673


,station,district,route,direction,type,year,hour,month,lanes,length,...,sd_speed,median_occup,avg_occup,sd_occup,days_observed,county,state_pm,abs_pm,latitude,longitude
0,1203481,12,91,W,ML,2023,0,January,5,0.684,...,1.444042,0.0210,0.021000,0.001910,13,59.0,R.49,15.229,33.859,-118.034
1,1203481,12,91,W,ML,2023,1,January,5,0.684,...,2.018726,0.0151,0.015077,0.001272,13,59.0,R.49,15.229,33.859,-118.034
2,1203481,12,91,W,ML,2023,2,January,5,0.684,...,1.258866,0.0140,0.014408,0.001303,13,59.0,R.49,15.229,33.859,-118.034
3,1203481,12,91,W,ML,2023,3,January,5,0.684,...,1.864066,0.0225,0.022008,0.001614,13,59.0,R.49,15.229,33.859,-118.034
4,1203481,12,91,W,ML,2023,4,January,5,0.684,...,1.476569,0.0470,0.046400,0.003032,13,59.0,R.49,15.229,33.859,-118.034
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77082,1221099,12,91,W,HV,2025,19,September,3,0.217,...,0.072501,0.0096,0.009600,0.001234,13,59.0,R18.1,36.513,33.871,-117.685
77083,1221099,12,91,W,HV,2025,20,September,3,0.217,...,0.055470,0.0059,0.006092,0.000708,13,59.0,R18.1,36.513,33.871,-117.685
77084,1221099,12,91,W,HV,2025,21,September,3,0.217,...,0.027735,0.0043,0.004362,0.000375,13,59.0,R18.1,36.513,33.871,-117.685
77085,1221099,12,91,W,HV,2025,22,September,3,0.217,...,0.040825,0.0027,0.002754,0.000320,13,59.0,R18.1,36.513,33.871,-117.685


In [70]:
# test API: query_traffic_by_route()
df_route = engine.query_traffic_by_route(
    route="5",
    direction="N",
    year=2025
)

display(df_route.head(10))

INFO:src.pems.query:Querying traffic data: route=5, direction=N, year=2025
INFO:src.pems.query:✅ Query returned 35,612 rows


,station,route,direction,hour,month,year,avg_flow,median_flow,avg_speed,median_speed,avg_occup,lanes,days_observed
0,1205045,5,N,0,April,2025,1403.000000,1437.0,68.380000,69.20,0.028260,6,5
1,1204950,5,N,0,April,2025,1364.928571,1350.0,70.964286,71.10,0.019036,6,14
2,1204384,5,N,0,April,2025,684.214286,682.0,68.321429,68.65,0.016357,5,14
3,1211107,5,N,0,April,2025,716.000000,716.5,68.092857,68.50,0.013671,5,14
4,1204409,5,N,0,April,2025,709.285714,703.5,69.342857,69.40,0.012471,5,14
5,1208365,5,N,0,April,2025,0.000000,0.0,65.000000,65.00,0.000000,2,14
6,1210926,5,N,0,April,2025,712.785714,705.5,68.842857,68.80,0.012314,5,14
7,1205607,5,N,0,April,2025,1565.857143,1553.0,68.921429,68.80,0.022536,7,14
8,1210992,5,N,0,April,2025,38.785714,38.0,64.957143,65.00,0.001293,2,14
9,1210071,5,N,0,April,2025,39.928571,39.5,64.957143,64.95,0.001993,2,14


In [71]:
# test API: get_kpi_summary()
kpi = engine.get_kpi_summary(year=2025)

print("2025 KPI summary:")
print(f"Total Route Number: {kpi['total_routes']}")
print(f"Total Station Number: {kpi['total_stations']}")
print(f"Average Flow: {kpi['avg_flow_overall']:.1f} veh/h")
print(f"Average Speed: {kpi['avg_speed_overall']:.1f} mph")
print(f"Total Observations: {kpi['total_observations']:,}")

INFO:src.pems.query:Fetching KPI summary for year 2025
INFO:src.pems.query:✅ KPI summary: 11.0 routes, 1259.0 stations


2025 KPI summary:
Total Route Number: 11.0
Total Station Number: 1259.0
Average Flow: 2203.4 veh/h
Average Speed: 62.1 mph
Total Observations: 2,783,553.0


---

## Step 7: Parameterized Query and SQL Injection Prevention

In [14]:
import duckdb
import pandas as pd
from pathlib import Path

# --- 1. Setup ---
DATA_DIR = Path("../data/processed")
PARQUET_FILE = DATA_DIR / "2025_station_hour_processed.parquet"

# --- 2. Connect to DuckDB ---
conn = duckdb.connect(database=":memory:")

# --- 3. Create a VIEW for the Parquet file (Corrected) ---
create_view_sql = f"CREATE OR REPLACE VIEW traffic_2025 AS SELECT * FROM read_parquet('{PARQUET_FILE}')"
conn.execute(create_view_sql)

# --- 4. Define the Query Function (This part was already correct and secure) ---
def query_route(route, direction, lane_type):
    """
    Queries the traffic_2025 view for a specific route, direction, and lane type.
    """
    # ✅ CORRECT: Use parameterized queries for DATA VALUES.
    sql = """
        SELECT *
        FROM traffic_2025
        WHERE route = ?
          AND direction = ?
          AND type = ?
    """
    return conn.execute(sql, [route, direction, lane_type]).df()

# --- 5. Execute the Query and Display Results ---
i5_n_ml_2025_df = query_route('5', 'N', 'ML')

display(i5_n_ml_2025_df)

,station,district,route,direction,type,year,hour,month,lanes,length,...,sd_speed,median_occup,avg_occup,sd_occup,days_observed,county,state_pm,abs_pm,latitude,longitude
0,1204198,12,5,N,ML,2025,0,January,5,0.913,...,1.824729,0.01350,0.013318,0.001787,11,59.0,.65,72.908,33.405,-117.598
1,1204198,12,5,N,ML,2025,1,January,5,0.913,...,3.216237,0.01160,0.011536,0.001562,11,59.0,.65,72.908,33.405,-117.598
2,1204198,12,5,N,ML,2025,2,January,5,0.913,...,2.807166,0.01000,0.009864,0.001264,11,59.0,.65,72.908,33.405,-117.598
3,1204198,12,5,N,ML,2025,3,January,5,0.913,...,2.589630,0.01220,0.012327,0.001520,11,59.0,.65,72.908,33.405,-117.598
4,1204198,12,5,N,ML,2025,4,January,5,0.913,...,3.711946,0.02120,0.020945,0.001178,11,59.0,.65,72.908,33.405,-117.598
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17292,1221232,12,5,N,ML,2025,8,September,5,0.650,...,0.359398,0.11520,0.116350,0.002816,4,59.0,4.7,76.958,33.450,-117.640
17293,1221232,12,5,N,ML,2025,9,September,5,0.650,...,0.739369,0.10940,0.108225,0.006152,4,59.0,4.7,76.958,33.450,-117.640
17294,1221232,12,5,N,ML,2025,10,September,5,0.650,...,1.061446,0.11585,0.117850,0.005548,4,59.0,4.7,76.958,33.450,-117.640
17295,1221232,12,5,N,ML,2025,11,September,5,0.650,...,0.556028,0.11800,0.118000,0.003812,4,59.0,4.7,76.958,33.450,-117.640


---

## 步驟 6: 清理資源

In [15]:
# close connection
conn.close()
engine.close()

INFO:src.pems.query:✅ DuckDB connection closed


---

## 🎓 學習總結

### 你學會了:

1. ✅ **建立 DuckDB 連線** (`duckdb.connect()`)
2. ✅ **直接查詢 Parquet** (`read_parquet()`)
3. ✅ **建立 VIEW** (簡化查詢)
4. ✅ **基礎 SQL 查詢** (WHERE, GROUP BY, ORDER BY)
5. ✅ **聚合函數** (AVG, MEDIAN, COUNT)
6. ✅ **使用專案 API** (`DuckDBQueryEngine`)

### 關鍵概念:

- **VIEW = 虛擬表格**: 不佔記憶體,延遲載入
- **Column Pruning**: 只讀取需要的欄位
- **Predicate Pushdown**: 在檔案層級過濾資料

### 下一步:

- Phase 2.2: R2 雲端查詢整合
- Phase 2.3: 查詢優化(LRU 快取)
- Phase 2.4: CLI 查詢工具
- Phase 2.5: Dashboard 整合